In [ ]:
# Setup — add src/ to path and import the project's own helpers
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data import load_data, RAW_DIR
from utils import (
    save_fig, plot_churn_distribution, plot_correlation_heatmap,
    plot_churn_by_feature, NUMERIC_COLS if False else None,
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_style('whitegrid')

print(f"Raw data dir: {RAW_DIR}")

In [ ]:
# 1. Load the data
df = load_data()
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
# 2. Target distribution
plot_churn_distribution(df)
save_fig('01_churn_distribution.png')
plt.show()

churn_rate = (df['Churn'] == 'Yes').mean()
print(f"Overall churn rate: {churn_rate:.2%}")

In [ ]:
# 3. Missing values & data health
print("Missing values per column:")
print(df.isna().sum())

# IBM Telco stores TotalCharges as object — coerce to numeric
if df['TotalCharges'].dtype == 'object':
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    print(f"\nAfter coercion — missing TotalCharges: {df['TotalCharges'].isna().sum()}")
    df['TotalCharges'] = df['TotalCharges'].fillna(0)

df.describe(include='all').T.head(15)

In [ ]:
# 4. Univariate & bivariate plots
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']

for col in ['Contract', 'InternetService', 'PaymentMethod', 'tenure', 'MonthlyCharges']:
    plot_churn_by_feature(df, col)
    save_fig(f'02_churn_by_{col}.png')
    plt.show()

In [ ]:
# 5. Correlation heatmap of numeric features
plot_correlation_heatmap(df, numeric_cols)
save_fig('03_correlation_heatmap.png')
plt.show()

In [ ]:
# 6. Group-level churn rates (the business-readable view)
def churn_rate_by(col):
    return (df.groupby(col)['Churn']
              .apply(lambda s: (s == 'Yes').mean())
              .sort_values(ascending=False))

for col in ['Contract', 'InternetService', 'PaymentMethod', 'PaperlessBilling', 'SeniorCitizen']:
    print(f"\nChurn rate by {col}:")
    print(churn_rate_by(col).round(3))

In [ ]:
# 7. Save a cleaned copy for the modelling notebook
from data import save_processed
processed_path = save_processed(df, name='churn_clean.csv')
print(f"Saved cleaned data → {processed_path}")